验证脉动阵列单次运行结果正确性

In [1]:
import numpy as np

# ==========================================
# 方法一：基础标量计算 (直接代数展开)
# ==========================================
a_scalar = 100 + 11*11 + 12*21 + 13*31 + 14*41
b_scalar = a_scalar + 21*11 + 22*21 + 23*31 + 24*41

print("=== 基础标量计算结果 ===")
print(f"a = {a_scalar}")
print(f"b = {b_scalar}\n")

# ==========================================
# 方法二：NumPy 向量化计算 (契合 NPU 硬件数据流设计)
# ==========================================
# 观察你的等式，可以将 [11, 21, 31, 41] 视为一组在多个周期内被复用的输入特征向量
vec_in = np.array([11, 21, 31, 41])

# 将 [11, 12, 13, 14] 视为第一轮的权重向量
weight_1 = np.array([11, 12, 13, 14])

# 将 [21, 22, 23, 24] 视为第二轮的权重向量
weight_2 = np.array([21, 22, 23, 24])

# 第一轮计算：偏置 (Bias) 为 100，进行向量点乘 (MAC 累加)
bias_initial = 100
a_numpy = bias_initial + np.dot(weight_1, vec_in)

# 第二轮计算：上一轮的结果 a_numpy 作为部分和 (PSum) 回流注入，继续累加
b_numpy = a_numpy + np.dot(weight_2, vec_in)

print("=== NumPy 向量化计算结果 ===")
print(f"a = {a_numpy}")
print(f"b = {b_numpy}")

=== 基础标量计算结果 ===
a = 1450
b = 3840

=== NumPy 向量化计算结果 ===
a = 1450
b = 3840


验证脉动阵列部分和一轮累加结果正确性

In [2]:
import numpy as np

# ==========================================
# 1. 初始化数据 (严格对应 Verilog 测试激励)
# ==========================================
# 权重矩阵 W[r][c]: 4x4
W = np.zeros((4, 4), dtype=np.int32)
for r in range(4):
    for col in range(4):
        W[r][col] = (r + 1) * 10 + (col + 1)

# 偏置向量 B[c]: 4
B = np.zeros(4, dtype=np.int32)
for col in range(4):
    B[col] = (col + 1) * 100

# 激活矩阵 A[r][cyc]: 4行 x 10组
A = np.zeros((4, 10), dtype=np.int32)
for cyc in range(10):
    for r in range(4):
        A[r][cyc] = (cyc + 1) * 10 + (r + 1)

# ==========================================
# 2. Python 版 expected_psum 软件复现
# ==========================================
def expected_psum(col, cyc_limit):
    """
    计算特定列(col)在经历了 cyc_limit 个周期的累加后的部分和
    """
    # 初始底座为 Bias
    total_sum = B[col]
    
    # 累加每一个 cycle 的 4 行乘积
    for cycle in range(cyc_limit):
        for r in range(4):
            # MAC 运算
            total_sum += A[r][cycle] * W[r][col]
            
    return total_sum

# ==========================================
# 3. 执行验证并打印结果
# ==========================================
print("=== 开始 Python 交叉验证 ===")

# 打印各列在不同周期（Vec 0 ~ Vec 10）的结果，对齐 Verilog 日志
for cyc in range(11):  # cyc 从 0 到 10
    results = [expected_psum(col, cyc) for col in range(4)]
    
    if cyc == 0:
        print(f"[Vec  0] (纯 Bias)     -> col0={results[0]:<5}, col1={results[1]:<5}, col2={results[2]:<5}, col3={results[3]:<5}")
    elif cyc == 1:
        print(f"[Vec  1] (Bias + 1次)  -> col0={results[0]:<5}, col1={results[1]:<5}, col2={results[2]:<5}, col3={results[3]:<5}")
    elif cyc == 10:
        print("-" * 65)
        print(f"[Vec 10] (最终完整结果) -> col0={results[0]:<5}, col1={results[1]:<5}, col2={results[2]:<5}, col3={results[3]:<5}")
        print("-" * 65)

=== 开始 Python 交叉验证 ===
[Vec  0] (纯 Bias)     -> col0=100  , col1=200  , col2=300  , col3=400  
[Vec  1] (Bias + 1次)  -> col0=1450 , col1=1600 , col2=1750 , col3=1900 
-----------------------------------------------------------------
[Vec 10] (最终完整结果) -> col0=60400, col1=62800, col2=65200, col3=67600
-----------------------------------------------------------------
